# Group Project Cryosphere

Earth Observation group project about a glacier in the Himalaya in Nepal. We try to measure glacier area, surface temperature variation, and wet/total snow area using satellite data from Google Earth Engine. We investigate these aspects on the time series of 2015 - 2026.

## Initializing

In [ ]:
import time
import rasterio
import numpy as np
import pandas as pd
from plotting import *
from gst_series import *
from sar_snow import *
from gst_pixel import *

In [ ]:
# run once at beginning, supply token
import ee
ee.Authenticate()

In [ ]:
GEE_PROJECT = "my-hello-world-508910"  # replace with your own project id
ee.Initialize(project=GEE_PROJECT)

In [ ]:
ROI = [[81.768322, 30.198924], [81.785316, 30.200185], [81.782656, 30.20241], [81.780596, 30.2079], [81.780252, 30.215613], [81.780424, 30.223327], [81.778965, 30.228741], [81.776733, 30.231707], [81.774759, 30.231781], [81.771927, 30.225403], [81.77021, 30.216503], [81.770382, 30.211237], [81.770296, 30.206416], [81.766691, 30.202707], [81.765575, 30.199962], [81.768322, 30.198924]]
AOI = ee.Geometry.Polygon(ROI)
aoi_center_lon_lat = AOI.centroid().coordinates().getInfo() # lon/lat
print(aoi_center_lon_lat)

In [ ]:
START = "2014-09-01"
END = "2026-12-31"

## Image Collections Analysis

Get a sense of how many Images we are working with, from what date they are and how much they are covered by clouds.

In [ ]:
s2_toa = ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(AOI).sort("system:time_start").set("collection_name", "Sentinel 2 Top of Atmosphere")
s2_boa = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(AOI).sort("system:time_start").set("collection_name", "Sentinel 2 Bottom of Atmosphere")

In [ ]:
plot_histogram_image_collection(s2_toa, "2015-09-01", "2027-01-01")
plot_histogram_image_collection(s2_boa, "2015-09-01", "2027-01-01")

In these Histograms we see that there Sentinel 2 provides data with few clouds from 2016 to 2026. Also, we see that cloud cover is very seasonal. Especially in summer, images with cloud coverage <30% are scarce. Although more and more images are provided throughout the time period, image count has been very consistent since 2019.

## Glacier area

Description here...

## Surface temperature variation

### GST series

In [ ]:
glacier_mask, mask_status = get_glacier_mask(roi=AOI)
print("Glacier mask source:", mask_status)

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
bin_img = dem.divide(BIN_M).floor().multiply(BIN_M).toInt().rename("bin") # 50 bzw. BIN_M grosse Höhenstufen
# show_map(dem=dem, roi=AOI, glacier_mask=glacier_mask) # needs debugging / checking!

In [ ]:
raw_gst = download(bin_img, glacier_mask, AOI, START, END, debug=False)
gst_half_month = process(raw_gst, debug=True)
ablation_trend = ablation_trend(gst_half_month, debug=True)
gst_half_month.head()

In [ ]:
fig_gst_observations_1 = plot_half_months(gst_half_month)
fig_gst_observations_2 = plot_half_months_2(gst_half_month)
fig_gst_observations_2.show()

In [ ]:
fig_gst_err = plot_gst_series(gst_half_month, errorbars=True)
fig_gst_noerr = plot_gst_series(gst_half_month, errorbars=False)
fig_gst_noerr.show()

In [ ]:
save_any_tables_and_figures(
    {"gst_raw_scene_bins": raw_gst,
     "gst_halfmonthly": gst_half_month,
     "gst_ablation_season": ablation_trend,
     "gst_obervations_1": fig_gst_observations_1,
     "gst_obervations_2": fig_gst_observations_2,
     "gst_err": fig_gst_err,
     "gst_noerr": fig_gst_noerr},
    save_path="results/gst_series/"
)

### GST per pixel

In [ ]:
scenes = download_zone_scenes(AOI, glacier_mask, START, END, debug=True)
half_monthly = process_zones(scenes, debug=True)

winters = winter_means(half_monthly)
print(winter_trends(winters).round(3))

In [ ]:
pixel_arrays = image_to_arrays(pixel_statistics(AOI, glacier_mask, START, END), AOI)
fig = plot_pixel_maps(pixel_arrays)
fig.show()

In [ ]:
point_series_df = download_point_series({"centre": (81.774, 30.220)}, START, END, debug=True)

In [ ]:
save_any_tables_and_figures(
    {"gst_scene_zone_sums": scenes,
     "gst_halfmonthly_winter": half_monthly,
     "gst_winter_means": winters,
     "gst_points": point_series_df,
     "gst_pixel_maps": fig},
    save_path = "results/gst_pixel/"
)

## Wet/total snow area

In [ ]:
df, bdf = all_stats(AOI, "2014-09-01", "2026-12-31", glacier_mask=glacier_mask, debug=True)

In [ ]:
SY = 2024
ORBIT = "ASCENDING"
MONTHS = (12, *range(1, 7))

day_ticks_all = ([32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335, 0],
                 ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
day_ticks = ([day_ticks_all[0][m - 1] for m in MONTHS],
             [day_ticks_all[1][m - 1] for m in MONTHS])
print(day_ticks)

In [ ]:
monthly_table = monthly_mean_table(df)

fig_snow_series = plot_sar_snow_series(df)
fig_snow_season = plot_sar_snow_season(df, day_ticks=DAY_TICKS)

fig_snow_elev = plot_sar_snow_elev(bdf, SY, ORBIT) # wet fraction by elevation band and date
fig_snow_diff_history = show_sar_snow_diff_history(AOI, SY, ORBIT, months=(12, *range(1, 12))) # backscatter-change histogram per month, threshold marked
fig_snow_maps = show_sar_snow_maps(AOI, SY, ORBIT, months=(12, *range(1, 12)), glacier_mask=glacier_mask) # monthly wet frequency and melt-onset day

In [ ]:
save_any_tables_and_figures(
    {"snow_scenes": df,
     "snow_elev_bands": bdf,
     "monthly_table": monthly_table,
     "sar_snow_series": fig_snow_series,
     "sar_snow_season": fig_snow_season},
    save_path="results/sar_snow/",
)
save_any_tables_and_figures(
    {f"sar_snow_elev_{SY}_{ORBIT}": fig_snow_elev,
     f"sar_snow_diff_history_{SY}_{ORBIT}": fig_snow_diff_history,
     f"sar_snow_maps_{SY}_{ORBIT}": fig_snow_maps},
    save_path="results/sar_snow/",
)